About Cryptography

- 대수학은 일방향 함수들로 가득한 보물창고이다!
    - NP-complete 문제를 활용한 암호시스템($P \neq NP$)를 가정함으로서 안정성을 담보 받음
        - TSP, 해밀턴 사이클, Knapsack, 최장거리 경로 등
    - 대부분 이산로그 문제들은 일방향 mapping이 이루어지는 구조체에서 발견됨!
        - 타원 곡선 유한체, 소인수분해(shor는 O((logN)^3)으로 양자 gate는 더 넓은 범위의 상상력 ㄱㄴ)
- 결정론적인 시스템(평문-암호문 일대일 대응)의 해독성에서 비롯된 확률적인 암호시스템(polly cracker)의 대두

Branch

- Symmetry-Key(use same key) / Assymetric-Key(use different key for enc/dec)
- Steganography()
- Quantum Cryptography

In [103]:
import random
import numpy as np
import math
import codecs

### Merkle-Hellman Knapsack System: Using NP-complete problem to form one-way function.
# Knapsack is NP-complete!
#   S = {s1, s2, ..., sn} and objective sum T, find subset S' such satisfying sum S'i = T.
#   -> with super-increasing sequenece

# Componenets
a = [] # super increasing sequence
noises = random.sample(list(range(10))*8, 20)
for i, noise in enumerate(noises):
    a.append(sum(a[:i])+noise)
m = a[-1] + random.randint(1, 10) # set modulus
w = 0 # power
for w in range(2, m):
    if math.gcd(w, m) == 1:
        break
w_inverse = 0
for w_inverse in range(m):
    if w * w_inverse % m == 1:
        break
public_key = list(map(lambda x: (x * w) % m, a))

make_binary = lambda string: bin(int(string.encode("utf-8").hex(), 16))[2:]
data = make_binary("HI")

data = [int(b) for b in data]
encoded = sum([(public_key[i] * b) % m for i, b in enumerate(data)])
print(f"Encode msg 'HI' into {encoded} with {','.join(map(str, public_key))}")

# w has its inverse, we could get data * w * a -> data * a (multiflying w^-1)
# in attacker perspective, factor C = sum(xi * b) -> NP-complete
decoded = (encoded * w_inverse) % m
decoded_data = []
for ai in reversed(a):
    if decoded >= ai:
        decoded -= ai
        decoded_data.append(1)
    else: decoded_data.append(0)
decoded_data = decoded_data[::-1]

binary_to_string = lambda binary: codecs.decode(hex(int("".join(map(str, binary)), 2))[2:], "hex").decode('utf-8')
print(f"Decoded {binary_to_string(decoded_data[:len(data)])}")

Encode msg 'HI' into 126591 with 0,6,27,60,108,225,435,861,1734,3462,6939,13860,27741,55464,110937,221859,443736,887469,591628,1183274
Decoded HI


In [ ]:
### Lattice Reduction Attack -> Merkle-Hellman is not safe!
# LLL finds the short Latice Vector to break it! in Polynomial time.


In [105]:
### ECDSA, who enables Digital Life.

from ecdsa.curves import SECP256k1
from ecdsa.ellipticcurve import Point
import hashlib

#Generator
x1 = 55066263022277343669578718895168534326250603453777594175500187360389116729240
y1 = 32670510020758816978083085130507043184471273380659243275938904335757337482424
G = Point(SECP256k1.curve, x1, y1)
print(G==G * (SECP256k1.order+1))
print("generator?", G == SECP256k1.generator)

n = SECP256k1.order

#Public key, Private key
k = 1002349230423
P = G*k

#Random point
l = 10
# r = 36322260242567644327577471914851727161017458705958127170915236715425819333073
# y = 113817104126258647026551196310596962231430747658282676626153792268748326724326
# R = Point(SECP256k1.curve, r, y)
R = G*l

#message hash
m = b"Don't Trust, Verify"
hash_obj = hashlib.sha256(m)
hash_hex = hash_obj.hexdigest()
z = int(hash_hex, 16)
print("message:", m)
print("message hash:", z)

#signature
s = (l+z*k) % n

#verification
print("valid signatre?")
print(l*G + z*P == s*G)

# Schnorr signature
### R + z P = s G ----> inside z, there are sign that I agree this signature. (ex. P-R-msg)
### (l1+l2+...) + z * (k1+k2+...) = s1+s2+...

True
generator? True
message: b"Don't Trust, Verify"
message hash: 112761469845056919304416565170674990599925418308225593267094103636319905743526
valid signatre?
True
